# 03 종합 — 표와 통계 (CPU)

pass@1 결과와 judge 결과를 **여기서 처음** `(model_key, task_id)`로 합칩니다. 표는 화면에 보이고 `analysis/`에 CSV로 저장됩니다.

| 표 | 내용 | 보고서에서의 역할 |
|---|---|---|
| T1 | 키별 pass@1, 95% CI(Wilson), 종료 사유·추출 실패·문법 오류, 실패 유형 | 생성 결과 기술 |
| T2 | 키 쌍별 pass@1 차이, 문제 단위 paired bootstrap CI, McNemar p | 모델 차이(참고용, 프로토콜 차이 명시) |
| T3 | judge별 상태(ok / 파싱 실패 / 잘림 / API 에러), 반환된 모델 버전, 비용 | judge 신뢰성 |
| T4 | judge의 correctness 점수 ↔ 실제 통과: AUC(문제 단위 cluster bootstrap CI), Spearman, 통과/실패별 평균, 오판율 | **주 질문 1: judge가 맞는가** |
| T5 | judge 간 일치도: 항목별 quadratic weighted kappa, Spearman, judge별 평균 점수(관대함) | **주 질문 2: judge끼리 일치하는가** |
| T6 | 같은 입력 재호출 시 흔들림: 항목별 완전 일치율, 평균 절대 차이 | judge 안정성 |
| T7 | halo 효과: 실패한 코드에 준 가독성·효율성·견고성 점수 vs 통과한 코드, correctness와의 상관 | 정답 없는 3항목의 해석 한계 + 편향 관찰 |
| T8 | 키 × judge 평균 점수 | 기술 통계 |

**해석 주의**: pass@1이 기준점(정답)입니다. 가독성·효율성·견고성은 정답이 없어 judge 간 일치도만 말할 수 있습니다. 1단계는 judge와 생성 모델 계열이 겹치지 않으므로 self-preference 해석은 하지 않습니다.

In [ ]:
RUN_NAME = "full"         # "smoke"로 바꾸면 smoke 결과로 표 형식만 확인 가능

import os, json, glob
from google.colab import drive
drive.mount("/content/drive")
RUN_DIR = f"/content/drive/MyDrive/code_eval_v4/{RUN_NAME}"

In [ ]:
# 로드 + 공통 함수
import itertools, math
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, binomtest
from sklearn.metrics import roc_auc_score, cohen_kappa_score

OUT_DIR = f"{RUN_DIR}/analysis"
os.makedirs(OUT_DIR, exist_ok=True)
CRITERIA = ["correctness", "readability", "efficiency", "robustness"]
N_BOOT, RNG = 2000, np.random.default_rng(0)
pd.set_option("display.width", 200, "display.max_columns", 30)

def read_jsonl(path):
    if not os.path.exists(path):
        return []
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

def show(name, df):
    df.to_csv(f"{OUT_DIR}/{name}.csv", index=True)
    print(f"\n### {name}")
    try:
        display(df)
    except NameError:
        print(df.to_string())

# 생성 (중복은 첫 레코드만)
gen_rows = []
for path in sorted(glob.glob(f"{RUN_DIR}/generations/*.jsonl")):
    seen = set()
    for r in read_jsonl(path):
        if r["task_id"] in seen:
            continue
        seen.add(r["task_id"])
        gen_rows.append({k: r[k] for k in ["task_id", "model_key", "mode", "finish_reason",
                                            "extraction_ok", "syntax_ok", "n_new_tokens"]})
gen = pd.DataFrame(gen_rows)
pas = pd.DataFrame(read_jsonl(f"{RUN_DIR}/pass_results.jsonl"))
pas["passed"] = pas["passed"].astype(int)

# judge: 같은 작업은 최신 레코드만
latest = {}
for r in read_jsonl(f"{RUN_DIR}/judge_results.jsonl"):
    latest[(r["model_key"], r["task_id"], r["judge"], r["rep"])] = r
jrecs = list(latest.values())
jall = pd.DataFrame([{**{k: r.get(k) for k in ["model_key", "task_id", "judge", "rep", "status",
                                               "judge_model_id", "model_returned", "latency_s"]},
                      "in_tok": (r.get("usage") or {}).get("input"),
                      "out_tok": (r.get("usage") or {}).get("output"),
                      **{c: (r.get("scores") or {}).get(c) for c in CRITERIA}} for r in jrecs])
JUDGES = sorted(jall["judge"].unique()) if len(jall) else []
ok = jall[(jall.status == "ok") & (jall.rep == 0)].merge(pas[["model_key", "task_id", "passed"]],
                                                        on=["model_key", "task_id"], how="inner")
print(f"생성 {len(gen)}건, pass 결과 {len(pas)}건, judge 레코드 {len(jall)}건 (분석용 ok·rep0 {len(ok)}건), judge {JUDGES}")

def wilson(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (c - h, c + h)

def cluster_boot(df, stat, cluster="task_id", n=N_BOOT):
    """문제 단위로 재표집(같은 문제의 여러 키·judge 점수는 서로 독립이 아니므로)"""
    groups = {g: d for g, d in df.groupby(cluster)}
    keys = list(groups)
    vals = []
    for _ in range(n):
        samp = pd.concat([groups[k] for k in RNG.choice(keys, len(keys), replace=True)])
        try:
            v = stat(samp)
        except ValueError:
            continue
        if v is not None and not np.isnan(v):
            vals.append(v)
    return (np.percentile(vals, 2.5), np.percentile(vals, 97.5)) if vals else (np.nan, np.nan)

def auc(df, col="correctness"):
    if df["passed"].nunique() < 2:
        return np.nan
    return roc_auc_score(df["passed"], df[col])

def qwk(a, b):
    if len(a) == 0:
        return np.nan
    return cohen_kappa_score(a, b, weights="quadratic", labels=[1, 2, 3, 4, 5])

In [ ]:
# T1 생성·pass@1 / T2 키 쌍 비교
g = gen.merge(pas, on=["model_key", "task_id"], how="left")
rows = []
for k, d in g.groupby("model_key"):
    n, kp = len(d), int(d.passed.sum())
    lo, hi = wilson(kp, n)
    fails = d[d.passed == 0].error_type.value_counts()
    rows.append({"model_key": k, "mode": d["mode"].iloc[0], "n": n, "passed": kp,
                 "pass@1": round(kp / n, 3), "CI95_lo": round(lo, 3), "CI95_hi": round(hi, 3),
                 "finish_stop": int((d.finish_reason == "stop").sum()),
                 "finish_eos": int((d.finish_reason == "eos").sum()),
                 "finish_length": int((d.finish_reason == "length").sum()),
                 "extraction_fail": int((~d.extraction_ok).sum()),
                 "syntax_fail": int((~d.syntax_ok).sum()),
                 "top_fail_types": ", ".join(f"{e}:{c}" for e, c in fails.head(4).items())})
T1 = pd.DataFrame(rows).set_index("model_key")
show("T1_pass_at_1", T1)

wide = pas.pivot(index="task_id", columns="model_key", values="passed").dropna()
rows = []
for a, b in itertools.combinations(wide.columns, 2):
    x, y = wide[a].values, wide[b].values
    diffs = [np.mean(x[i] - y[i]) for i in (RNG.integers(0, len(x), len(x)) for _ in range(N_BOOT))]
    n10, n01 = int(((x == 1) & (y == 0)).sum()), int(((x == 0) & (y == 1)).sum())
    p = binomtest(n10, n10 + n01, 0.5).pvalue if n10 + n01 else 1.0
    rows.append({"A": a, "B": b, "n_tasks": len(x), "diff(A-B)": round(x.mean() - y.mean(), 3),
                 "CI95_lo": round(np.percentile(diffs, 2.5), 3), "CI95_hi": round(np.percentile(diffs, 97.5), 3),
                 "A만통과": n10, "B만통과": n01, "McNemar_p": round(p, 4)})
T2 = pd.DataFrame(rows)
show("T2_pairwise_pass", T2)
print("※ raw(본문 이어쓰기)와 chat(함수 전체 생성) 키 사이 비교는 프로토콜 차이가 섞임")

In [ ]:
# T3 judge 신뢰성·비용 / T4 judge ↔ 실제 통과 (주 질문 1)
if len(jall):
    T3 = jall.groupby("judge").agg(
        calls=("status", "size"),
        ok=("status", lambda s: int((s == "ok").sum())),
        parse_error=("status", lambda s: int((s == "parse_error").sum())),
        truncated=("status", lambda s: int((s == "truncated").sum())),
        api_error=("status", lambda s: int((s == "api_error").sum())),
        mean_in_tok=("in_tok", "mean"), mean_out_tok=("out_tok", "mean"),
        mean_latency_s=("latency_s", "mean"),
        model_returned=("model_returned", lambda s: ", ".join(sorted(set(map(str, s.dropna()))))),
    )
    T3["fail_rate"] = ((T3.parse_error + T3.truncated) / (T3.calls - T3.api_error).clip(lower=1)).round(3)
    show("T3_judge_reliability", T3.round(1))

def validity_row(name, d):
    lo, hi = cluster_boot(d, auc)
    fp = d[d.passed == 0]
    fn = d[d.passed == 1]
    return {"judge": name, "n": len(d), "n_pass": int(d.passed.sum()),
            "AUC": round(auc(d), 3), "AUC_CI95_lo": round(lo, 3), "AUC_CI95_hi": round(hi, 3),
            "spearman": round(spearmanr(d.correctness, d.passed).statistic, 3),
            "mean_corr_pass": round(fn.correctness.mean(), 2), "mean_corr_fail": round(fp.correctness.mean(), 2),
            "실패인데_4점이상": round((fp.correctness >= 4).mean(), 3) if len(fp) else np.nan,
            "통과인데_2점이하": round((fn.correctness <= 2).mean(), 3) if len(fn) else np.nan}

if len(ok):
    rows = [validity_row(j, d) for j, d in ok.groupby("judge")]
    # 합의 judge: 모든 judge가 ok인 항목에서 correctness 평균
    cons = ok.pivot_table(index=["model_key", "task_id"], columns="judge", values="correctness").dropna()
    if cons.shape[1] > 1:
        c = cons.mean(axis=1).rename("correctness").reset_index().merge(
            pas[["model_key", "task_id", "passed"]], on=["model_key", "task_id"])
        rows.append(validity_row(f"평균({cons.shape[1]}개 judge)", c))
    T4 = pd.DataFrame(rows).set_index("judge")
    show("T4_judge_validity", T4)

    rows = []                                   # 키별로 나눠 본 AUC (특정 생성 모델에서만 잘 맞는지)
    for (j, k), d in ok.groupby(["judge", "model_key"]):
        rows.append({"judge": j, "model_key": k, "n": len(d), "pass_rate": round(d.passed.mean(), 3),
                     "AUC": round(auc(d), 3)})
    show("T4b_validity_by_model", pd.DataFrame(rows).pivot(index="judge", columns="model_key", values="AUC"))

In [ ]:
# T5 judge 간 일치도 (주 질문 2) / T6 재호출 흔들림
if len(ok) and len(JUDGES) > 1:
    rows = []
    for c in CRITERIA:
        w = ok.pivot_table(index=["model_key", "task_id"], columns="judge", values=c)
        for a, b in itertools.combinations(w.columns, 2):
            d = w[[a, b]].dropna().astype(int)
            rows.append({"criterion": c, "judge_A": a, "judge_B": b, "n": len(d),
                         "QWK": round(qwk(d[a], d[b]), 3),
                         "spearman": round(spearmanr(d[a], d[b]).statistic, 3),
                         "exact_agree": round((d[a] == d[b]).mean(), 3),
                         "mean_A-B": round((d[a] - d[b]).mean(), 2)})
    T5 = pd.DataFrame(rows)
    show("T5_inter_judge", T5)
    show("T5b_inter_judge_QWK_summary", T5.pivot_table(index=["judge_A", "judge_B"], columns="criterion", values="QWK"))
    show("T5c_judge_mean_scores", ok.groupby("judge")[CRITERIA].mean().round(2))

rep = jall[jall.status == "ok"]
r0 = rep[rep.rep == 0].set_index(["model_key", "task_id", "judge"])[CRITERIA]
r1 = rep[rep.rep == 1].set_index(["model_key", "task_id", "judge"])[CRITERIA]
both = r0.join(r1, lsuffix="_0", rsuffix="_1", how="inner")
if len(both):
    rows = []
    for j, d in both.groupby(level="judge"):
        row = {"judge": j, "n": len(d)}
        for c in CRITERIA:
            row[f"{c}_일치율"] = round((d[f"{c}_0"] == d[f"{c}_1"]).mean(), 3)
            row[f"{c}_평균절대차"] = round((d[f"{c}_0"] - d[f"{c}_1"]).abs().mean(), 2)
        rows.append(row)
    show("T6_judge_stability", pd.DataFrame(rows).set_index("judge"))
else:
    print("\nT6: 재호출(rep=1) 결과 없음 — full 모드에서 생성됨")

In [ ]:
# T7 halo 효과 / T8 키 × judge 평균
OTHER = ["readability", "efficiency", "robustness"]
if len(ok):
    rows = []
    for j, d in ok.groupby("judge"):
        row = {"judge": j}
        for c in OTHER:
            row[f"{c}_통과"] = round(d[d.passed == 1][c].mean(), 2)
            row[f"{c}_실패"] = round(d[d.passed == 0][c].mean(), 2)
            row[f"{c}~correctness_rho"] = round(spearmanr(d[c], d.correctness).statistic, 3)
        rows.append(row)
    show("T7_halo", pd.DataFrame(rows).set_index("judge"))
    print("읽는 법: 실패한 코드의 가독성·견고성 점수가 높게 유지되면 항목을 분리해 보지 못한다는 뜻,"
          " correctness와의 상관이 매우 높으면 항목 간 구분이 약하다는 뜻(halo)")

    show("T8_mean_by_model_judge",
         ok.groupby(["model_key", "judge"])[CRITERIA].mean().round(2))

print(f"\nCSV 저장 위치: {OUT_DIR}")